# 06 — Emotion Detector (Wav2Vec2 → VAD Regression)
## TeluguVoiceBridge v2 — Constrained Hardware Plan

**Model:** `facebook/wav2vec2-base` with VAD regression head  
**Output:** Valence, Arousal, Dominance (3 continuous values in [-1, 1])  
**Training data:** RAVDESS (8 emotions → VAD mapping)  
**Loss:** MSE + CCC (Concordance Correlation Coefficient)  
**Target:** CCC ≥ 0.55 on held-out test

### Why Continuous VAD?
- Discrete labels lose intensity: "slightly irritated" ≠ "furious"
- VAD provides smooth conditioning signal for TTS FiLM layers
- Better interpolation between emotional states at inference time

---
## 6.1 — Setup & Config

In [1]:
import os, gc, pathlib, time, json, csv, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
from omegaconf import OmegaConf

BASE = pathlib.Path(os.getcwd())
CONFIG = OmegaConf.load(BASE / "configs" / "emotion.yaml")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(CONFIG.training.device)
print(f"Device: {DEVICE}")
print(f"Config:\n{OmegaConf.to_yaml(CONFIG)}")

Device: cuda
Config:
model:
  name: facebook/wav2vec2-base
  freeze_layers: 6
  vad_output_dim: 3
training:
  batch_size: 4
  epochs: 20
  learning_rate: 0.0001
  weight_decay: 0.0001
  device: cuda
  save_total_limit: 2
loss:
  type: mse_ccc
  ccc_weight: 0.5
target_metrics:
  ccc: 0.55



In [2]:
# Ensure GPU is clean
torch.cuda.empty_cache()
gc.collect()
vram = torch.cuda.memory_allocated() / 1e9
print(f"VRAM in use: {vram:.2f} GB")

VRAM in use: 0.00 GB


---
## 6.2 — VAD Mapping (Russell & Mehrabian)

RAVDESS uses 8 discrete emotion labels. We convert these to continuous VAD vectors.

In [3]:
# Russell & Mehrabian emotion-to-VAD mapping
EMOTION_TO_VAD = {
    "neutral":  [0.0,  0.0,  0.0],
    "calm":     [0.2, -0.3,  0.0],
    "happy":    [0.8,  0.5,  0.3],
    "sad":      [-0.6, -0.4, -0.4],
    "angry":    [-0.5,  0.7,  0.6],
    "fearful":  [-0.5,  0.6, -0.5],
    "disgust":  [-0.6,  0.2,  0.2],
    "surprised":[ 0.2,  0.7,  0.0],
}

# RAVDESS emotion codes
RAVDESS_CODES = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised",
}

print("Emotion → VAD mapping:")
for emo, vad in EMOTION_TO_VAD.items():
    print(f"  {emo:10s} → V={vad[0]:+.1f}  A={vad[1]:+.1f}  D={vad[2]:+.1f}")

Emotion → VAD mapping:
  neutral    → V=+0.0  A=+0.0  D=+0.0
  calm       → V=+0.2  A=-0.3  D=+0.0
  happy      → V=+0.8  A=+0.5  D=+0.3
  sad        → V=-0.6  A=-0.4  D=-0.4
  angry      → V=-0.5  A=+0.7  D=+0.6
  fearful    → V=-0.5  A=+0.6  D=-0.5
  disgust    → V=-0.6  A=+0.2  D=+0.2
  surprised  → V=+0.2  A=+0.7  D=+0.0


---
## 6.3 — Wav2Vec2 + VAD Head Model

In [11]:
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

class EmotionVADModel(nn.Module):
    """
    Wav2Vec2-base with VAD regression head.
    
    Architecture:
    - Wav2Vec2 backbone (freeze first N layers)
    - Mean pooling over time
    - Dropout(0.1) → Linear(768, 128) → ReLU → Dropout(0.1) → Linear(128, 3) → Tanh
    - Output: [valence, arousal, dominance] in [-1, 1]
    """
    def __init__(self, model_name, freeze_layers=6, vad_dim=3):
        super().__init__()
        self.backbone = Wav2Vec2Model.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size  # 768
        
        # Freeze feature extractor
        self.backbone.feature_extractor._freeze_parameters()
        
        # Freeze first N encoder layers
        for i, layer in enumerate(self.backbone.encoder.layers):
            if i < freeze_layers:
                for param in layer.parameters():
                    param.requires_grad = False
        
        # VAD regression head
        self.head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, vad_dim),
            nn.Tanh(),  # Constrain to [-1, 1]
        )
    
    def forward(self, input_values, attention_mask=None):
        outputs = self.backbone(
            input_values=input_values,
            attention_mask=attention_mask,
        )
        hidden_states = outputs.last_hidden_state  # [batch, time_downsampled, 768]
        
        # Mean pooling — use the model's output lengths to build a proper mask
        # The backbone downsamples the raw audio, so attention_mask doesn't match hidden_states
        pooled = hidden_states.mean(dim=1)  # [batch, 768]
        
        vad = self.head(pooled)  # [batch, 3]
        return vad

# Load model
print(f"Loading {CONFIG.model.name}...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(CONFIG.model.name)
emotion_model = EmotionVADModel(
    model_name=CONFIG.model.name,
    freeze_layers=CONFIG.model.freeze_layers,
    vad_dim=CONFIG.model.vad_output_dim,
).to(DEVICE)

# Count parameters
total = sum(p.numel() for p in emotion_model.parameters())
trainable = sum(p.numel() for p in emotion_model.parameters() if p.requires_grad)
print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,} ({100*trainable/total:.1f}%)")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("✓ EmotionVADModel loaded.")

Loading facebook/wav2vec2-base...


/home/nibiru/.conda/envs/ml_env/lib/python3.11/site-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


Total params:     94,470,531
Trainable params: 47,742,851 (50.5%)
VRAM: 0.82 GB
✓ EmotionVADModel loaded.


---
## 6.4 — Loss Functions (MSE + CCC)

In [5]:
def concordance_correlation_coefficient(pred, target):
    """
    Concordance Correlation Coefficient (CCC).
    Standard evaluation metric for emotion regression in SER community.
    
    CCC = 2 * cov(pred, target) / (var(pred) + var(target) + (mean(pred) - mean(target))^2)
    
    Returns per-dimension CCC: [ccc_v, ccc_a, ccc_d]
    """
    mean_pred = pred.mean(dim=0)
    mean_target = target.mean(dim=0)
    var_pred = pred.var(dim=0)
    var_target = target.var(dim=0)
    
    # Covariance
    cov = ((pred - mean_pred) * (target - mean_target)).mean(dim=0)
    
    # CCC
    numerator = 2 * cov
    denominator = var_pred + var_target + (mean_pred - mean_target) ** 2
    ccc = numerator / (denominator + 1e-8)
    
    return ccc  # [3] for V, A, D


class MSE_CCC_Loss(nn.Module):
    """
    Combined MSE + CCC loss for VAD regression.
    L = MSE + ccc_weight * (1 - mean(CCC))
    """
    def __init__(self, ccc_weight=0.5):
        super().__init__()
        self.mse = nn.MSELoss()
        self.ccc_weight = ccc_weight
    
    def forward(self, pred, target):
        mse_loss = self.mse(pred, target)
        
        # CCC loss (need at least 2 samples)
        if pred.shape[0] > 1:
            ccc = concordance_correlation_coefficient(pred, target)
            ccc_loss = 1.0 - ccc.mean()
        else:
            ccc_loss = torch.tensor(0.0, device=pred.device)
        
        total = mse_loss + self.ccc_weight * ccc_loss
        return total, mse_loss, ccc_loss

criterion = MSE_CCC_Loss(ccc_weight=CONFIG.loss.ccc_weight).to(DEVICE)
print(f"✓ MSE + CCC loss (ccc_weight={CONFIG.loss.ccc_weight})")

✓ MSE + CCC loss (ccc_weight=0.5)


---
## 6.5 — Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

class EmotionVADDataset(Dataset):
    """
    Emotion dataset with VAD labels + augmentation.
    Loads audio, resamples to 16kHz, extracts features, returns VAD target.

    Augmentations (train only):
    - Speed perturbation: randomly stretch/compress by 0.9-1.1x
    - Pitch shift: shift pitch by -2 to +2 semitones
    - Additive noise: small Gaussian noise (SNR ~40dB)
    These make the emotion detector robust across speakers and languages.
    """
    def __init__(self, manifest_path, feature_extractor, base_dir, split="train",
                 sr=16000, max_dur=10.0, augment=True):
        df = pd.read_csv(manifest_path)
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.feature_extractor = feature_extractor
        self.base_dir = pathlib.Path(base_dir)
        self.sr = sr
        self.max_samples = int(max_dur * sr)
        self.augment = augment and (split == "train")
        
        print(f"  {split}: {len(self.df)} clips (augment={self.augment})")
    
    def _augment_waveform(self, wav):
        """Apply random augmentations to waveform."""
        # Speed perturbation (0.9x to 1.1x)
        if random.random() < 0.5:
            speed_factor = random.uniform(0.9, 1.1)
            indices = np.arange(0, len(wav), speed_factor)
            indices = indices[indices < len(wav)].astype(int)
            wav = wav[indices]
        
        # Pitch shift via resampling trick (-2 to +2 semitones)
        if random.random() < 0.3:
            semitones = random.uniform(-2, 2)
            rate = 2 ** (semitones / 12)
            resampled = torchaudio.functional.resample(
                torch.from_numpy(wav).float().unsqueeze(0),
                int(self.sr * rate), self.sr
            ).squeeze(0).numpy()
            wav = resampled
        
        # Additive noise (low level, ~40dB SNR)
        if random.random() < 0.3:
            noise_level = random.uniform(0.001, 0.005)
            wav = wav + noise_level * np.random.randn(len(wav))
        
        return wav
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = self.base_dir / row["audio_path"]
        
        # Load audio
        wav, sr = sf.read(str(audio_path))
        if sr != self.sr:
            wav = torchaudio.functional.resample(
                torch.from_numpy(wav).float().unsqueeze(0), sr, self.sr
            ).squeeze(0).numpy()
        
        # Truncate
        if len(wav) > self.max_samples:
            wav = wav[:self.max_samples]
        
        # Apply augmentation (train only)
        if self.augment:
            wav = self._augment_waveform(wav)
        
        # Extract features
        features = self.feature_extractor(
            wav, sampling_rate=self.sr, return_tensors="pt", padding=True
        )
        input_values = features.input_values.squeeze(0)
        
        # VAD target
        vad = torch.tensor([
            float(row["valence"]),
            float(row["arousal"]),
            float(row["dominance"]),
        ], dtype=torch.float32)
        
        return input_values, vad


def collate_emotion(batch):
    """Pad input_values to same length in batch."""
    input_values = [item[0] for item in batch]
    vad_labels = torch.stack([item[1] for item in batch])
    
    # Pad to max length in batch
    max_len = max(v.shape[0] for v in input_values)
    padded = torch.zeros(len(input_values), max_len)
    attention_mask = torch.zeros(len(input_values), max_len, dtype=torch.long)
    
    for i, v in enumerate(input_values):
        padded[i, :v.shape[0]] = v
        attention_mask[i, :v.shape[0]] = 1
    
    return padded, attention_mask, vad_labels

print("✓ EmotionVADDataset (with augmentation) and collate function defined.")

✓ EmotionVADDataset and collate function defined.


In [7]:
# Load datasets
EMO_MANIFEST = BASE / "data" / "metadata" / "emotion_manifest.csv"

if not EMO_MANIFEST.exists():
    print("⚠ Emotion manifest not found. Run notebook 02 first.")
    existing = BASE.parent / "data" / "metadata" / "emotion_manifest.csv"
    if existing.exists():
        import shutil
        shutil.copy2(existing, EMO_MANIFEST)
        print(f"  ✓ Copied from {existing}")

print("Loading emotion datasets...")
train_dataset = EmotionVADDataset(EMO_MANIFEST, feature_extractor, BASE, split="train")
val_dataset = EmotionVADDataset(EMO_MANIFEST, feature_extractor, BASE, split="val")

BATCH_SIZE = CONFIG.training.batch_size

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, collate_fn=collate_emotion,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, collate_fn=collate_emotion,
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

Loading emotion datasets...
  train: 1137 clips
  val: 120 clips

Train batches: 285
Val batches:   30


---
## 6.6 — Training Setup

In [12]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, emotion_model.parameters()),
    lr=CONFIG.training.learning_rate,
    weight_decay=CONFIG.training.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

scaler = torch.amp.GradScaler("cuda")

EPOCHS = CONFIG.training.epochs
CKPT_DIR = BASE / "checkpoints" / "emotion_detector"
LOG_FILE = BASE / "logs" / "emotion_training_log.csv"

with open(LOG_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss",
                    "ccc_v", "ccc_a", "ccc_d", "ccc_mean", "lr", "time_sec"])

print(f"Epochs:        {EPOCHS}")
print(f"Batch size:    {BATCH_SIZE}")
print(f"LR:            {CONFIG.training.learning_rate}")
print(f"Save limit:    {CONFIG.training.save_total_limit}")

Epochs:        20
Batch size:    4
LR:            0.0001
Save limit:    2


---
## 6.7 — Evaluation Function

In [9]:
@torch.no_grad()
def evaluate_emotion(model, val_loader, device):
    """
    Evaluate emotion model: compute val loss & per-dimension CCC.
    """
    model.eval()
    all_preds = []
    all_targets = []
    total_loss = 0.0
    n = 0
    
    for input_values, attention_mask, vad_labels in val_loader:
        input_values = input_values.to(device)
        attention_mask = attention_mask.to(device)
        vad_labels = vad_labels.to(device)
        
        with torch.amp.autocast("cuda"):
            preds = model(input_values, attention_mask)
            loss, _, _ = criterion(preds, vad_labels)
        
        total_loss += loss.item()
        all_preds.append(preds.float().cpu())
        all_targets.append(vad_labels.float().cpu())
        n += 1
    
    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    
    # Compute CCC per dimension
    ccc = concordance_correlation_coefficient(all_preds, all_targets)
    avg_loss = total_loss / max(n, 1)
    
    # Also compute MSE per dimension
    mse_per_dim = F.mse_loss(all_preds, all_targets, reduction='none').mean(dim=0)
    
    model.train()
    return avg_loss, ccc, mse_per_dim

print("✓ Evaluation function ready.")

✓ Evaluation function ready.


---
## 6.8 — Training Loop

In [13]:
# ═══════════════════════════════════════════════════
# MAIN TRAINING LOOP — Emotion Detector (Wav2Vec2 → VAD)
# ═══════════════════════════════════════════════════

best_ccc_mean = -1.0
patience_counter = 0
PATIENCE = 8
t_start = time.time()

print("="*60)
print("Starting Emotion Detector Training")
print("="*60)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    emotion_model.train()
    epoch_loss = 0.0
    n_batches = 0
    
    for batch_idx, (input_values, attention_mask, vad_labels) in enumerate(train_loader):
        input_values = input_values.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        vad_labels = vad_labels.to(DEVICE)
        
        with torch.amp.autocast("cuda"):
            preds = emotion_model(input_values, attention_mask)
            loss, mse_l, ccc_l = criterion(preds, vad_labels)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(emotion_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        n_batches += 1
        
        if (batch_idx + 1) % 20 == 0:
            vram = torch.cuda.memory_allocated() / 1e9
            print(f"  Epoch {epoch} | Batch {batch_idx+1}/{len(train_loader)} | "
                  f"Loss: {loss.item():.4f} | VRAM: {vram:.1f}GB", end="\r")
    
    avg_train_loss = epoch_loss / max(n_batches, 1)
    
    # ─── Validation ───
    val_loss, val_ccc, val_mse = evaluate_emotion(emotion_model, val_loader, DEVICE)
    ccc_mean = val_ccc.mean().item()
    
    epoch_time = time.time() - epoch_start
    lr = optimizer.param_groups[0]["lr"]
    
    print(f"\nEpoch {epoch:>2d}/{EPOCHS} | Train: {avg_train_loss:.4f} | "
          f"Val: {val_loss:.4f} | CCC: V={val_ccc[0]:.3f} A={val_ccc[1]:.3f} "
          f"D={val_ccc[2]:.3f} (mean={ccc_mean:.3f}) | {epoch_time:.0f}s")
    
    # Log
    with open(LOG_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([epoch, f"{avg_train_loss:.4f}", f"{val_loss:.4f}",
                        f"{val_ccc[0]:.4f}", f"{val_ccc[1]:.4f}", f"{val_ccc[2]:.4f}",
                        f"{ccc_mean:.4f}", f"{lr:.2e}", f"{epoch_time:.0f}"])
    
    # LR scheduling on mean CCC (maximize)
    scheduler.step(ccc_mean)
    
    # ─── Save best ───
    if ccc_mean > best_ccc_mean:
        best_ccc_mean = ccc_mean
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": emotion_model.state_dict(),
            "ccc": val_ccc.tolist(),
            "ccc_mean": ccc_mean,
        }, CKPT_DIR / "best_model.pt")
        print(f"  ★ New best CCC mean: {ccc_mean:.4f} → saved")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch}")
            break
    
    # Save last
    torch.save({
        "epoch": epoch,
        "model_state_dict": emotion_model.state_dict(),
    }, CKPT_DIR / "last_model.pt")
    
    torch.cuda.empty_cache()
    gc.collect()

total_time = time.time() - t_start
print(f"\n{'='*60}")
print(f"Training complete!")
print(f"Best CCC mean: {best_ccc_mean:.4f} (target: ≥ {CONFIG.target_metrics.ccc})")
print(f"Total time: {total_time/60:.1f} min")
print(f"{'='*60}")

Starting Emotion Detector Training
  Epoch 1 | Batch 280/285 | Loss: 0.4746 | VRAM: 1.4GB
Epoch  1/20 | Train: 0.5093 | Val: 0.5803 | CCC: V=0.815 A=0.795 D=0.770 (mean=0.793) | 22s
  ★ New best CCC mean: 0.7933 → saved
  Epoch 2 | Batch 280/285 | Loss: 0.1914 | VRAM: 1.4GB
Epoch  2/20 | Train: 0.3305 | Val: 0.6109 | CCC: V=0.688 A=0.787 D=0.756 (mean=0.743) | 21s
  Epoch 3 | Batch 280/285 | Loss: 0.1389 | VRAM: 1.4GB
Epoch  3/20 | Train: 0.2794 | Val: 0.5508 | CCC: V=0.892 A=0.903 D=0.829 (mean=0.875) | 21s
  ★ New best CCC mean: 0.8746 → saved
  Epoch 4 | Batch 280/285 | Loss: 0.3446 | VRAM: 1.4GB
Epoch  4/20 | Train: 0.2459 | Val: 0.5648 | CCC: V=0.856 A=0.806 D=0.895 (mean=0.852) | 21s
  Epoch 5 | Batch 280/285 | Loss: 0.1928 | VRAM: 1.4GB
Epoch  5/20 | Train: 0.2331 | Val: 0.5940 | CCC: V=0.737 A=0.819 D=0.837 (mean=0.798) | 21s
  Epoch 6 | Batch 280/285 | Loss: 0.6583 | VRAM: 1.4GB
Epoch  6/20 | Train: 0.2135 | Val: 0.5753 | CCC: V=0.804 A=0.848 D=0.853 (mean=0.835) | 21s
  Epoch

---
## 6.9 — Final Test Evaluation

In [14]:
# Load best model
best_ckpt = torch.load(CKPT_DIR / "best_model.pt", map_location=DEVICE, weights_only=False)
emotion_model.load_state_dict(best_ckpt["model_state_dict"])
print(f"Loaded best model from epoch {best_ckpt['epoch']}")

# Test set
test_dataset = EmotionVADDataset(EMO_MANIFEST, feature_extractor, BASE, split="test")
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, collate_fn=collate_emotion,
)

test_loss, test_ccc, test_mse = evaluate_emotion(emotion_model, test_loader, DEVICE)
test_ccc_mean = test_ccc.mean().item()

print(f"\n{'='*40}")
print(f"FINAL TEST RESULTS")
print(f"{'='*40}")
print(f"CCC Valence:   {test_ccc[0]:.4f}")
print(f"CCC Arousal:   {test_ccc[1]:.4f}")
print(f"CCC Dominance: {test_ccc[2]:.4f}")
print(f"CCC Mean:      {test_ccc_mean:.4f} (target: ≥ {CONFIG.target_metrics.ccc})")
print(f"MSE (V/A/D):   {test_mse[0]:.4f} / {test_mse[1]:.4f} / {test_mse[2]:.4f}")

ccc_pass = test_ccc_mean >= CONFIG.target_metrics.ccc
print(f"\nCCC target: {'✓ PASS' if ccc_pass else '✗ FAIL'}")

# Save results
results = {
    "model": CONFIG.model.name,
    "best_epoch": best_ckpt["epoch"],
    "test_ccc_valence": float(test_ccc[0]),
    "test_ccc_arousal": float(test_ccc[1]),
    "test_ccc_dominance": float(test_ccc[2]),
    "test_ccc_mean": test_ccc_mean,
    "training_time_min": round(total_time / 60, 2),
}
with open(CKPT_DIR / "training_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {CKPT_DIR / 'training_results.json'}")

Loaded best model from epoch 10
  test: 178 clips

FINAL TEST RESULTS
CCC Valence:   0.7291
CCC Arousal:   0.8180
CCC Dominance: 0.7231
CCC Mean:      0.7567 (target: ≥ 0.55)
MSE (V/A/D):   0.1578 / 0.0931 / 0.1052

CCC target: ✓ PASS

Results saved to /home/nibiru/Documents/sem6project/Speech2/pipeline_v2/checkpoints/emotion_detector/training_results.json


---
## 6.10 — Inference Demo

In [15]:
# Test inference on a sample
emotion_model.eval()

# Try inference on a random test sample
test_df = pd.read_csv(EMO_MANIFEST)
test_df = test_df[test_df["split"] == "test"]

if len(test_df) > 0:
    sample = test_df.sample(1).iloc[0]
    audio_path = BASE / sample["audio_path"]
    
    if audio_path.exists():
        wav, sr = sf.read(str(audio_path))
        if sr != 16000:
            wav = torchaudio.functional.resample(
                torch.from_numpy(wav).float().unsqueeze(0), sr, 16000
            ).squeeze(0).numpy()
        
        features = feature_extractor(wav, sampling_rate=16000, return_tensors="pt")
        input_values = features.input_values.to(DEVICE)
        
        with torch.no_grad():
            vad_pred = emotion_model(input_values).cpu().squeeze(0)
        
        print(f"Audio:      {sample['audio_path']}")
        print(f"Emotion:    {sample.get('emotion', 'unknown')}")
        print(f"Target VAD: V={sample['valence']:+.2f} A={sample['arousal']:+.2f} D={sample['dominance']:+.2f}")
        print(f"Pred   VAD: V={vad_pred[0]:+.2f} A={vad_pred[1]:+.2f} D={vad_pred[2]:+.2f}")
    else:
        print(f"Sample audio not found: {audio_path}")
else:
    print("No test samples available.")

# Inference speed
test_audio = torch.randn(1, 5 * 16000).to(DEVICE)  # 5s
times = []
for _ in range(10):
    t0 = time.time()
    with torch.no_grad():
        _ = emotion_model(test_audio)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    times.append(time.time() - t0)

avg_ms = np.mean(times) * 1000
print(f"\nInference time (5s audio): {avg_ms:.1f} ms")
print(f"Target: ≤ 500 ms → {'✓ PASS' if avg_ms <= 500 else '✗ FAIL'}")

Audio:      data/processed/emotion_train/emo_000841.wav
Emotion:    neutral
Target VAD: V=+0.00 A=+0.00 D=+0.00
Pred   VAD: V=-0.04 A=+0.04 D=+0.01

Inference time (5s audio): 14.9 ms
Target: ≤ 500 ms → ✓ PASS


In [ ]:
# Cleanup GPU
del emotion_model, optimizer, scaler
torch.cuda.empty_cache()
gc.collect()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("✓ GPU freed for next phase.")

VRAM after cleanup: 1.77 GB
✓ GPU freed for next phase.


: 

---
## ✓ Notebook 06 Complete

**What we accomplished:**
- Fine-tuned Wav2Vec2-base with VAD regression head
- MSE + CCC combined loss (CCC weight 0.5)
- Russell & Mehrabian emotion→VAD mapping for RAVDESS
- Frozen feature extractor + first 6 encoder layers
- Per-dimension CCC evaluation (V, A, D)
- **NEW: Data augmentation** — speed perturbation, pitch shifting, and additive noise for cross-speaker/cross-language robustness

**Why this matters for voice preservation:**
The VAD predictions from this model drive the FiLM conditioning layers in TTS. 
Better emotion detection → more accurate FiLM modulation → TTS output with correct 
emotional character → more natural voice preservation across languages.

**Target:** CCC ≥ 0.55  
**Next:** Open `07_tts_finetuning.ipynb`